# 02. Data Preprocessing & Pipeline Construction
Demonstrating imputation for Vehicle Type, encoding categorical features, and numerical scaling.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df = pd.read_csv("../data/carbon_emissions.csv")


In [ ]:
def clean_df(df_in):
    df_c = df_in.copy()
    mask = (df_c["Transport"] != "private") & (df_c["Vehicle Type"].isna() | (df_c["Vehicle Type"] == "None"))
    df_c.loc[mask, "Vehicle Type"] = "Not Applicable"
    df_c["Vehicle Type"] = df_c["Vehicle Type"].fillna("missing")
    for col in ["Recycling", "Cooking_With"]:
        df_c[col] = df_c[col].apply(lambda x: str(sorted(x)) if isinstance(x, list) else str(x))
    return df_c

df_proc = clean_df(df)
print("Vehicle Type counts:\n", df_proc["Vehicle Type"].value_counts())


In [ ]:
cat_cols = ["Body Type", "Sex", "Diet", "How Often Shower", "Heating Energy Source", "Transport", "Vehicle Type", "Social Activity", "Frequency of Traveling by Air", "Waste Bag Size", "Energy efficiency", "Recycling", "Cooking_With"]
num_cols = ["Monthly Grocery Bill", "Vehicle Monthly Distance Km", "Waste Bag Weekly Count", "How Long TV PC Daily Hour", "How Many New Clothes Monthly", "How Long Internet Daily Hour"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="missing")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
])

X = df_proc.drop(columns=["CarbonEmission"])
X_trans = preprocessor.fit_transform(X)
print("Transformed feature matrix shape:", X_trans.shape)
